In [9]:
import requests            # lets Python ask a website for data
import pandas as pd        # tables of data (like Excel inside Python)
import numpy as np         # maths on lists and columns of numbers

LATITUDE = 28.61           # north-south position of your chosen place
LONGITUDE = 77.21          # east-west position of your chosen place

START_DATE = "2018-01-01"  # first day of data we want
END_DATE = "2026-10-31"    # last day of data we want

print("Setup done")                                  # shows a message so we know the cell ran
print("Location:", LATITUDE, LONGITUDE)              # shows the place we chose
print("Dates:", START_DATE, "to", END_DATE)          # shows the date range

Setup done
Location: 28.61 77.21
Dates: 2018-01-01 to 2026-10-31


In [10]:
url = "https://archive-api.open-meteo.com/v1/archive"   # address of the free Open-Meteo history service

params = {                                               # the "order form" we send to the service
    "latitude": LATITUDE,                                # place: north-south, from Step 1
    "longitude": LONGITUDE,                              # place: east-west, from Step 1
    "start_date": START_DATE,                            # first day we want
    "end_date": END_DATE,                                # last day we want
    "hourly": "temperature_2m,relative_humidity_2m,dew_point_2m,surface_pressure,precipitation",  # the 5 measurements we want, one value per hour
    "timezone": "auto",                                  # use the local time of the chosen place
}

response = requests.get(url, params=params)             # sends the order and receives the reply
print("Status code:", response.status_code)             # 200 means success

data = response.json()                                   # turns the reply into a Python dictionary
df = pd.DataFrame(data["hourly"])                        # puts the hourly values into a table
df["time"] = pd.to_datetime(df["time"])                  # changes the time column from plain text to real dates
df = df.set_index("time")                                # makes time the row label, so each row is one hour

print("Table size (rows, columns):", df.shape)           # how many hours and how many measurements
print(df.head())                                         # shows the first 5 rows
print("Missing values per column:")                       # heading for the next line
print(df.isna().sum())                                   # counts empty cells in each column

df.to_csv("weather_raw.csv")                             # saves the table as a file in Colab

Status code: 200
Table size (rows, columns): (77424, 5)
                     temperature_2m  relative_humidity_2m  dew_point_2m  \
time                                                                      
2018-01-01 00:00:00            11.8                    76           7.7   
2018-01-01 01:00:00             9.0                    86           6.8   
2018-01-01 02:00:00             8.2                    87           6.1   
2018-01-01 03:00:00             7.9                    87           5.9   
2018-01-01 04:00:00             8.1                    85           5.8   

                     surface_pressure  precipitation  
time                                                  
2018-01-01 00:00:00             988.5            0.0  
2018-01-01 01:00:00             988.3            0.0  
2018-01-01 02:00:00             987.5            0.0  
2018-01-01 03:00:00             987.1            0.0  
2018-01-01 04:00:00             987.0            0.0  
Missing values per column:
temper

In [11]:
print("Sorted by time:", df.index.is_monotonic_increasing)   # True means rows are in order from oldest to newest
print("Duplicate times:", df.index.duplicated().sum())        # counts repeated hours; we want 0

RAIN_LIMIT = 0.1                                              # an hour with 0.1 mm or more counts as a rain hour

rain_hour = (df["precipitation"] >= RAIN_LIMIT).astype(int)  # 1 if it rained in that hour, else 0

next_1 = rain_hour.shift(-1)                                  # rain value of the next hour, placed on this row
next_2 = rain_hour.shift(-2)                                  # rain value of 2 hours later
next_3 = rain_hour.shift(-3)                                  # rain value of 3 hours later

df["rain_next_3h"] = pd.concat([next_1, next_2, next_3], axis=1).max(axis=1)  # 1 if any of the 3 future hours had rain

df = df.iloc[:-3]                                             # drop the last 3 rows, because their future is unknown
df["rain_next_3h"] = df["rain_next_3h"].astype(int)           # change from 1.0/0.0 to whole numbers 1/0

print("Rows now:", len(df))                                   # should be 3 less than before
print("Share of rows with rain coming:")                      # heading for the next line
print(df["rain_next_3h"].value_counts(normalize=True))        # fraction of 0s and 1s

df.to_csv("weather_with_target.csv")                          # save progress

Sorted by time: True
Duplicate times: 0
Rows now: 77421
Share of rows with rain coming:
rain_next_3h
0    0.841232
1    0.158768
Name: proportion, dtype: float64


In [12]:
feat = pd.DataFrame(index=df.index)                          # empty table that will hold our features, same hourly rows as df

feat["pressure"] = df["surface_pressure"]                    # current pressure in hPa
feat["humidity"] = df["relative_humidity_2m"]                # current humidity in %
feat["temp"] = df["temperature_2m"]                          # current temperature in °C

feat["dew_gap"] = df["temperature_2m"] - df["dew_point_2m"]  # gap between air temp and dew point; small gap = air is nearly saturated = rain likely

for h in [1, 3, 6, 12]:                                      # look back 1, 3, 6 and 12 hours
    feat[f"pressure_change_{h}h"] = df["surface_pressure"].diff(h)   # pressure now minus pressure h hours ago

for h in [1, 3, 6]:                                          # look back 1, 3 and 6 hours
    feat[f"humidity_change_{h}h"] = df["relative_humidity_2m"].diff(h)  # humidity now minus humidity h hours ago

feat["temp_change_3h"] = df["temperature_2m"].diff(3)        # temperature now minus 3 hours ago
feat["dew_gap_change_3h"] = feat["dew_gap"].diff(3)          # is the air getting closer to saturation?

rain_hour_now = (df["precipitation"] >= RAIN_LIMIT).astype(int)  # 1 if it rained in each hour (past and present)
feat["rain_last_3h"] = rain_hour_now.rolling(3).sum()        # how many of the last 3 hours had rain
feat["rain_last_6h"] = rain_hour_now.rolling(6).sum()        # how many of the last 6 hours had rain

hour = df.index.hour                                          # hour of the day, 0 to 23
feat["hour_sin"] = np.sin(2 * np.pi * hour / 24)             # hour turned into a circle position (part 1)
feat["hour_cos"] = np.cos(2 * np.pi * hour / 24)             # hour turned into a circle position (part 2)

month = df.index.month                                        # month of the year, 1 to 12
feat["month_sin"] = np.sin(2 * np.pi * month / 12)           # month turned into a circle position (part 1)
feat["month_cos"] = np.cos(2 * np.pi * month / 12)           # month turned into a circle position (part 2)

data = feat.copy()                                            # final table starts as the features
data["rain_next_3h"] = df["rain_next_3h"]                     # add the target column from Step 3
data = data.dropna()                                          # remove the first few rows that have no past to look back at

print("Rows:", len(data))                                     # how many rows remain
print("Features:", data.shape[1] - 1)                         # number of feature columns (target not counted)

print("Correlation of each feature with rain coming:")        # heading for the next line
print(data.corr()["rain_next_3h"].drop("rain_next_3h").sort_values())  # which features move together with the target

data.to_csv("features.csv")                                   # save progress

Rows: 77409
Features: 19
Correlation of each feature with rain coming:
pressure              -0.291397
month_sin             -0.261745
dew_gap               -0.242513
month_cos             -0.225371
hour_cos              -0.120148
humidity_change_6h    -0.074706
humidity_change_3h    -0.059605
pressure_change_1h    -0.038577
pressure_change_3h    -0.030270
humidity_change_1h    -0.028702
pressure_change_6h     0.012749
dew_gap_change_3h      0.048922
temp_change_3h         0.059832
hour_sin               0.078655
pressure_change_12h    0.095234
temp                   0.142827
humidity               0.283090
rain_last_6h           0.571702
rain_last_3h           0.595252
Name: rain_next_3h, dtype: float64


In [13]:
X = data.drop(columns=["rain_next_3h"])                       # X = all the feature columns (the clues)
y = data["rain_next_3h"]                                      # y = the target column (the answer we want to predict)

X_train = X.loc[:"2023-12-31"]                                # training clues: everything up to the end of 2023
y_train = y.loc[:"2023-12-31"]                                # training answers for the same period

X_val = X.loc["2024-01-01":"2024-12-31"]                      # validation clues: the year 2024, used to compare and tune models
y_val = y.loc["2024-01-01":"2024-12-31"]                      # validation answers for 2024

X_test = X.loc["2025-01-01":]                                 # test clues: 2025 onwards, kept untouched until the very end
y_test = y.loc["2025-01-01":]                                 # test answers for 2025 onwards

for name, part_X, part_y in [("Train", X_train, y_train),     # loop over the three parts to print a summary of each
                             ("Validation", X_val, y_val),
                             ("Test", X_test, y_test)]:
    print(name)                                               # name of the part
    print("  Rows:", len(part_X))                             # number of hours in this part
    print("  From:", part_X.index.min(), "to", part_X.index.max())  # first and last time in this part
    print("  Rain share:", round(part_y.mean(), 3))           # fraction of rows where rain was coming

Train
  Rows: 52572
  From: 2018-01-01 12:00:00 to 2023-12-31 23:00:00
  Rain share: 0.154
Validation
  Rows: 8784
  From: 2024-01-01 00:00:00 to 2024-12-31 23:00:00
  Rain share: 0.153
Test
  Rows: 16053
  From: 2025-01-01 00:00:00 to 2026-10-31 20:00:00
  Rain share: 0.179


In [14]:
from sklearn.linear_model import LogisticRegression             # a simple model that draws a straight-line boundary
from sklearn.preprocessing import StandardScaler                # puts all features on a similar scale
from sklearn.pipeline import make_pipeline                      # chains steps (scale, then model) into one object
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score  # our scoring tools

def report(name, y_true, y_pred, y_score):                      # a helper that prints all 4 scores for any model
    print(name)                                                  # model name
    print("  Precision:", round(precision_score(y_true, y_pred, zero_division=0), 3))  # of the rain alarms, how many were right
    print("  Recall:   ", round(recall_score(y_true, y_pred), 3))                      # of the real rain events, how many we caught
    print("  F1:       ", round(f1_score(y_true, y_pred), 3))                          # one number mixing precision and recall
    print("  ROC-AUC:  ", round(roc_auc_score(y_true, y_score), 3))                    # how well the model ranks rain hours above dry hours

# Baseline 1: always say "no rain"
pred_zero = np.zeros(len(y_val), dtype=int)                     # a column of zeros, one per validation hour
report("Always no rain", y_val, pred_zero, pred_zero)           # score it

# Baseline 2: "if it rained in the last 3 hours, say rain will come"
pred_persist = (X_val["rain_last_3h"] > 0).astype(int)          # 1 if any of the last 3 hours had rain, else 0
report("Persistence rule", y_val, pred_persist, X_val["rain_last_3h"])  # score it, using the rain count as the ranking score

# Baseline 3: logistic regression
logreg = make_pipeline(                                          # build a 2-step pipeline
    StandardScaler(),                                            # step 1: scale every feature to a similar range
    LogisticRegression(max_iter=1000, class_weight="balanced"),  # step 2: the model, giving rain hours extra weight
)
logreg.fit(X_train, y_train)                                     # learn from the training data only
pred_lr = logreg.predict(X_val)                                  # predict 0 or 1 for each validation hour
prob_lr = logreg.predict_proba(X_val)[:, 1]                      # predict the probability of rain for each hour
report("Logistic regression", y_val, pred_lr, prob_lr)           # score it

Always no rain
  Precision: 0.0
  Recall:    0.0
  F1:        0.0
  ROC-AUC:   0.5
Persistence rule
  Precision: 0.696
  Recall:    0.696
  F1:        0.696
  ROC-AUC:   0.828
Logistic regression
  Precision: 0.534
  Recall:    0.899
  F1:        0.67
  ROC-AUC:   0.937


In [15]:
import lightgbm as lgb                                          # the LightGBM library (already installed in Colab)

model = lgb.LGBMClassifier(                                     # create the model with these settings
    n_estimators=1000,                                          # build up to 1000 small trees
    learning_rate=0.03,                                         # each tree makes only a small correction (slow and careful)
    num_leaves=31,                                              # limit on how complex each tree can be
    min_child_samples=50,                                       # each branch needs at least 50 hours of data (stops memorising noise)
    subsample=0.8,                                              # each tree sees a random 80% of the rows
    subsample_freq=1,                                           # pick that random 80% again for every tree
    colsample_bytree=0.8,                                       # each tree sees a random 80% of the features
    random_state=42,                                            # fixes the randomness so results repeat
    verbose=-1,                                                 # hides the long training messages
)

model.fit(                                                      # train the model
    X_train, y_train,                                           # learn from the training data
    eval_set=[(X_val, y_val)],                                  # after each tree, check the score on validation data
    eval_metric="auc",                                          # the score to watch is ROC-AUC
    callbacks=[lgb.early_stopping(50, verbose=False)],          # stop if validation score has not improved for 50 trees
)

print("Trees actually used:", model.best_iteration_)            # how many trees were kept after early stopping

prob_gb = model.predict_proba(X_val)[:, 1]                      # rain probability for every validation hour
pred_gb = (prob_gb >= 0.5).astype(int)                          # say "rain" if probability is 50% or more
report("LightGBM (threshold 0.5)", y_val, pred_gb, prob_gb)     # score it with the helper from Step 6

print("\nThreshold scan on validation data:")                   # heading for the table below
print("threshold  precision  recall  f1")                       # column names
best_t, best_f1 = 0.5, 0                                        # remember the best threshold found so far
for t in np.arange(0.10, 0.91, 0.05):                           # try thresholds 0.10, 0.15, ... 0.90
    p = (prob_gb >= t).astype(int)                              # predictions at this threshold
    pr = precision_score(y_val, p, zero_division=0)             # precision at this threshold
    rc = recall_score(y_val, p)                                 # recall at this threshold
    f1 = f1_score(y_val, p)                                     # F1 at this threshold
    print(f"{t:8.2f}  {pr:9.3f}  {rc:6.3f}  {f1:5.3f}")         # print one row of the table
    if f1 > best_f1:                                            # if this is the best F1 so far
        best_t, best_f1 = t, f1                                 # remember this threshold

print("\nBest threshold:", round(best_t, 2), "with F1", round(best_f1, 3))  # the winner

Trees actually used: 207
LightGBM (threshold 0.5)
  Precision: 0.737
  Recall:    0.714
  F1:        0.725
  ROC-AUC:   0.947

Threshold scan on validation data:
threshold  precision  recall  f1
    0.10      0.504   0.947  0.657
    0.15      0.536   0.915  0.676
    0.20      0.570   0.889  0.695
    0.25      0.599   0.853  0.704
    0.30      0.622   0.816  0.706
    0.35      0.652   0.793  0.715
    0.40      0.679   0.766  0.720
    0.45      0.705   0.736  0.720
    0.50      0.737   0.714  0.725
    0.55      0.761   0.679  0.718
    0.60      0.787   0.643  0.708
    0.65      0.800   0.592  0.681
    0.70      0.819   0.535  0.647
    0.75      0.843   0.450  0.587
    0.80      0.856   0.348  0.495
    0.85      0.898   0.248  0.389
    0.90      0.927   0.122  0.215

Best threshold: 0.5 with F1 0.725


In [16]:
# Part A: final scores on the test set (data the model has never been tuned on)
print("=== TEST SET RESULTS ===")                                       # heading

pred_p = (X_test["rain_last_3h"] > 0).astype(int)                       # persistence rule predictions on test
report("Persistence rule", y_test, pred_p, X_test["rain_last_3h"])      # score it

pred_l = logreg.predict(X_test)                                         # logistic regression predictions on test
prob_l = logreg.predict_proba(X_test)[:, 1]                             # logistic regression probabilities on test
report("Logistic regression", y_test, pred_l, prob_l)                   # score it

prob_t = model.predict_proba(X_test)[:, 1]                              # LightGBM probabilities on test
pred_t = (prob_t >= 0.5).astype(int)                                    # use the threshold 0.5 chosen on validation (no re-tuning)
report("LightGBM", y_test, pred_t, prob_t)                              # score it

# Part B: which features did LightGBM rely on?
gain = model.booster_.feature_importance(importance_type="gain")        # how much each feature improved the model's predictions
imp = pd.Series(gain, index=X_train.columns)                            # label each number with its feature name
imp = (imp / imp.sum() * 100).sort_values(ascending=False)              # convert to percent of total, biggest first
print("\n=== FEATURE IMPORTANCE (% of total) ===")                      # heading
print(imp.round(1))                                                     # print the ranking

# Part C: the physics check, a model with no rain-history features
drop_cols = ["rain_last_3h", "rain_last_6h"]                            # the two "it rained recently" features
Xtr2 = X_train.drop(columns=drop_cols)                                  # training clues without them
Xva2 = X_val.drop(columns=drop_cols)                                    # validation clues without them
Xte2 = X_test.drop(columns=drop_cols)                                   # test clues without them

model2 = lgb.LGBMClassifier(                                            # same settings as before
    n_estimators=1000, learning_rate=0.03, num_leaves=31,
    min_child_samples=50, subsample=0.8, subsample_freq=1,
    colsample_bytree=0.8, random_state=42, verbose=-1,
)
model2.fit(                                                             # train on the reduced features
    Xtr2, y_train,
    eval_set=[(Xva2, y_val)], eval_metric="auc",
    callbacks=[lgb.early_stopping(50, verbose=False)],
)

prob2 = model2.predict_proba(Xte2)[:, 1]                                # probabilities on test
pred2 = (prob2 >= 0.5).astype(int)                                      # same 0.5 threshold
print("\n=== PHYSICS-ONLY MODEL (no rain history) ===")                 # heading
report("LightGBM without rain history", y_test, pred2, prob2)           # score it

imp2 = pd.Series(model2.booster_.feature_importance(importance_type="gain"), index=Xtr2.columns)  # importance for this model
imp2 = (imp2 / imp2.sum() * 100).sort_values(ascending=False)          # percent, biggest first
print("\nTop 8 features of the physics-only model:")                    # heading
print(imp2.head(8).round(1))                                            # print the top 8

model.booster_.save_model("rain_model.txt")                             # save the main model for the app in Step 9

=== TEST SET RESULTS ===
Persistence rule
  Precision: 0.63
  Recall:    0.63
  F1:        0.63
  ROC-AUC:   0.783
Logistic regression
  Precision: 0.503
  Recall:    0.805
  F1:        0.619
  ROC-AUC:   0.898
LightGBM
  Precision: 0.708
  Recall:    0.574
  F1:        0.634
  ROC-AUC:   0.907

=== FEATURE IMPORTANCE (% of total) ===
rain_last_3h           33.9
rain_last_6h           22.7
humidity                7.2
dew_gap                 5.1
humidity_change_6h      3.8
month_cos               3.7
pressure                3.7
temp_change_3h          3.4
hour_sin                3.2
temp                    3.2
hour_cos                2.2
month_sin               1.8
pressure_change_1h      1.2
humidity_change_1h      1.2
pressure_change_12h     1.0
dew_gap_change_3h       0.9
pressure_change_3h      0.7
pressure_change_6h      0.7
humidity_change_3h      0.3
dtype: float64

=== PHYSICS-ONLY MODEL (no rain history) ===
LightGBM without rain history
  Precision: 0.668
  Recall:    0.484
  